In [2]:
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
import math


from energies.bend import Bend
from energies.bend_twist import BendTwist
from energies.gravity import Gravity
from energies.random import RandomForce
from energies.twist import Twist
from math_util.rotation import RotationUtil, Quaternion
from math_util.vectors import Vector
from rod.RodHelixConverter import RodHelixConverter
from rod.helix import Helix
from rod.helix_util import HelixUtil
from rod.preprocess import Preprocess
from rod.rod_generator import RodGenerator
from rod.rod_util import RodUtil
from solver.sim import Sim
from visualization.visualizer import Visualizer
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)

### Visualization Tools

In [23]:
def strands_to_one_objs(strands: np.ndarray, frame_idx: int, output_file: str = None, y_up: bool = True):
    output_file = f"output/obj/obj_{frame_idx}.obj" if output_file is None else output_file
    Visualizer.clear_output_file(output_file)
    vertex_offset = 1
    for strand in strands:
        pos = strand[:, :3]
        vertex_offset = Visualizer.to_simple_obj(pos=pos, output_file=output_file, init_offset=vertex_offset, y_up=y_up)
    return

def visualize_rod_plotly(poses, title="Rod Visualization"):
    """
    Interactive 3D visualization of rod centerline and material frames using Plotly.

    Parameters:
    - pos: list of (N, 3) arrays of positions (centerlines).
    """

    fig = go.Figure()

    # Plot centerline
    for i in range(len(poses)):
        fig.add_trace(go.Scatter3d(
            x=poses[i][:, 0], y=poses[i][:, 1], z=poses[i][:, 2],
            mode='lines',
            name='Centerline',
            line=dict(color='black', width=3)
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        legend=dict(x=0, y=1)
    )

    fig.show()

### Different Twist Mechanisms: w/ angular velocities and global rotations

In [ ]:
# Adding twist through generalized coordinates
def add_twist(pos, theta, twist_indices=None, twist_values=None, twist_rate=0.8):
    """
    Injecting twist at randomly selected vertices.

    Parameters:
    - pos: (N, 3) array of vertex positions.
    - theta: twist angles (N-1,) — passed through RodHelixConverter.
    - twist_rate: fraction of vertices to twist.

    Returns:
    - pos_out, theta_out: positions and twists after applying twist.
    """
    helix = RodHelixConverter.rod_to_helix(pos=pos, theta=theta)
    num_vertices = len(pos)
    num_twist_indices = len(helix.q) // 3

    # Determine how many total twist applications to apply
    if twist_values is None and twist_indices is None:
        total_twists = int(twist_rate * num_vertices)
        twist_indices = np.random.choice(np.arange(num_twist_indices), size=total_twists, replace=False)
        twist_values = np.random.uniform(-np.pi, np.pi, size=len(twist_indices))

        twist_indices = np.array(twist_indices)
        twist_values = np.array(twist_values)
        twist_values = np.degrees(twist_values)

    sorted_order = np.argsort(twist_indices)
    twist_indices = twist_indices[sorted_order]
    twist_values = twist_values[sorted_order]

    # Apply  twist: each twist affects this index and all after
    for i, idx in enumerate(twist_indices):
        delta_twist = twist_values[i] # convert to degrees
        helix.q[3 * idx] += delta_twist  # apply to this edge

    return RodHelixConverter.helix_to_rod(helix=helix)

# Adding twist through global rotations
def add_global_twist(pos, theta, twist_indices=None, twist_values=None, twist_rate=0.8, n_iters=50):
    '''
    Apply twist to a rod and project to maintain segment inextensibility
    Apply twist at designated twist indices by corresponding twist value. Both twist_indices and twist_values must be arrays of the same length

    Parameters:
    - pos: (N, 3) array of vertex positions.
    - theta: passed through unchanged - PROBLEM if we want to simulate on these strands
    - twist_indices: array of indices at which to apply twist. If None, randomly chosen
    - twist_values: array of twist angles (in radians) to apply at the corresponding twist_indices
    - twist_rate: fraction of segments to twist, if twist_indices not provided
    - n_iters: number of projection iterations for enforcing inextensibility

    Returns:
    - pos_rotated: (N, 3) rotated and projected positions
    - theta: unchanged
    '''
    pos_rotated = pos.copy()
    num_vertices = len(pos)

    # Rest lengths of each segment
    rest_lengths = np.linalg.norm(pos[1:] - pos[:-1], axis=1)

    max_twist_points = num_vertices - 1
    total_twists = int(twist_rate * max_twist_points)

    # Generate twist indices/values if not provided
    if twist_indices is None:
        twist_indices = np.random.choice(np.arange(max_twist_points), size=total_twists, replace=False)
    if twist_values is None:
        twist_values = np.random.uniform(-0.05, 0.05, size=len(twist_indices))

    for i, twist_index in enumerate(twist_indices):
        if twist_index >= num_vertices - 1:
            continue

        twist_angle = twist_values[i]
        axis = pos[twist_index + 1] - pos[twist_index]
        axis_norm = np.linalg.norm(axis)
        if axis_norm < 1e-8:
            continue
        axis = axis / axis_norm

        # Rodrigues' rotation formula
        K = np.array([
            [0, -axis[2], axis[1]],
            [axis[2], 0, -axis[0]],
            [-axis[1], axis[0], 0]
        ])
        I = np.eye(3)
        R = I + np.sin(twist_angle) * K + (1 - np.cos(twist_angle)) * (K @ K)

        origin = pos[twist_index]
        for j in range(twist_index + 1, num_vertices):
            vec = pos_rotated[j] - origin
            pos_rotated[j] = origin + R @ vec

    # Project to enforce inextensibility
    for _ in range(n_iters):
        for i in range(num_vertices - 1):
            p1, p2 = pos_rotated[i], pos_rotated[i + 1]
            edge = p2 - p1
            current_len = np.linalg.norm(edge)
            if current_len < 1e-8:
                continue
            rest_len = rest_lengths[i]
            correction = 0.5 * (1 - rest_len / current_len) * edge
            pos_rotated[i] += correction
            pos_rotated[i + 1] -= correction

    return pos_rotated, theta

def project_inextensibility(pos, rest_lengths, max_iters=10, tol=1e-6):
    for _ in range(max_iters):
        diffs = pos[1:] - pos[:-1]
        seg_lengths = np.linalg.norm(diffs, axis=1)
        errors = seg_lengths - rest_lengths

        if np.max(np.abs(errors)) < tol:
            break

        corrections = (errors / seg_lengths)[:, None] * diffs * 0.5
        pos[:-1] += corrections
        pos[1:] -= corrections

    return pos

def in_to_meters(x):
    inches_to_meters = 0.0254
    return x * inches_to_meters

### Visualize generalized & global twist at `index=100` of ~.03 radians for two identical curls

In [72]:
from rod.rod_generator import RodGenerator
from rod.RodHelixConverter import RodHelixConverter
from rod.helix_util import HelixUtil
from scipy.spatial.transform import Rotation
from math_util.rotation import RotationUtil

L = 12 # 12 inches
curl_wavelength = 1 # inch
n_points = 256
height_scale = L / n_points # vertical displacement per segment
indices_per_curl = curl_wavelength / height_scale
ang_curl_freq_default = (2 * np.pi) / indices_per_curl
f = ang_curl_freq_default
r, tf = 0.5, 0.8

pos, theta = RodGenerator.example_rod(n=n_points, curl_radius=in_to_meters(r), curl_frequency=f, height_scale=in_to_meters(height_scale))
rest_lengths = np.linalg.norm(pos[1:] - pos[:-1], axis=1)

# adding twist with generalized coords
twist_indices = np.array([100])
rest_lengths = np.linalg.norm(pos[1:] - pos[:-1], axis=1)
twist_values = np.array([90 / rest_lengths[twist_indices]])
pos, theta = add_twist(pos, theta, twist_indices, twist_values)

# adding twist with global rotations
pos_g, theta_g = add_global_twist(pos, theta, twist_indices=np.array([101]), twist_values= np.array([np.radians(np.pi / 2)]))

visualize_rod_plotly([pos, pos_g])

/var/folders/08/9rbxcpbs2rl1znd03wn5hz6m0000gn/T/ipykernel_28528/3516219993.py:35: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

